In [1]:
import os

from google import genai
from google.genai import types
from google.genai import errors

# Set GEMINI_API_KEY and STOCK_TICKER in the environment before running this notebook.
symbol = os.environ.get("STOCK_TICKER", "").strip().upper()
if not symbol:
    raise RuntimeError("Set STOCK_TICKER before running enterprise.ipynb")

api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY before running enterprise.ipynb")

client = genai.Client(api_key=api_key)

# Ordered list of models for fallback to handle rate limits
fallback_models = [
    "gemini-3.8-flash",
    "gemini-3.0-flash",       # Standardized "Gemini 3 Flash"
    "gemini-2.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite"
]

description_response = None
model_errors = []

for model_name in fallback_models:
    try:
        description_response = client.models.generate_content(
            model=model_name,
            contents=(
                f"Describe the company identified by stock ticker {symbol}. "
                "If the ticker is unknown or ambiguous, say so instead of guessing."
            ),
            config=types.GenerateContentConfig(
                system_instruction="Answer professionally in no more than 200 words. Describe the company's business concisely, without bullet points. You may include historical facts.",
                temperature=0.2,
            ),
        )
        break
        
    except errors.APIError as e:
        model_errors.append(f"{model_name}: {e}")
        continue
    except Exception as e:
        model_errors.append(f"{model_name}: {e}")
        continue

# Check if we exhausted all models without getting a response
if description_response is None:
    error_details = "\n".join(model_errors)
    raise RuntimeError(
        f"Failed to retrieve a description for {symbol} after exhausting all fallback models.\n"
        f"Model errors:\n{error_details}"
    )

description = (description_response.text or "").strip()
if not description:
    raise RuntimeError(f"Gemini returned an empty description for {symbol}")

print(description)

The ticker TSM identifies Taiwan Semiconductor Manufacturing Company Limited (TSMC), the world’s largest dedicated semiconductor foundry. Founded in 1987 by Morris Chang in Hsinchu, Taiwan, TSMC pioneered the pure-play foundry business model. By focusing exclusively on manufacturing integrated circuits for third-party clients rather than designing its own branded chips, TSMC catalyzed the rise of the global "fabless" semiconductor industry. 

Today, the company is a critical linchpin of the global technology supply chain, providing advanced manufacturing capabilities to major technology firms such as Apple, NVIDIA, and AMD. TSMC leads the industry in manufacturing cutting-edge microchips using advanced process nodes, including 5-nanometer and 3-nanometer technologies. Its manufactured semiconductors power a vast array of modern applications, including consumer electronics, artificial intelligence accelerators, high-performance computing systems, automotive technologies, and telecommuni